In [1]:
import pandas as pd
import glob
import os

In [2]:
data_folder = "data/SP500_Data_10Y/"  
all_files = glob.glob(os.path.join(data_folder, "*.csv"))
print(f"Found {len(all_files)} CSV files")

Found 501 CSV files


In [3]:
def load_one_file(filepath):
    df = pd.read_csv(filepath, skiprows=[1, 2], header=0)
    df.columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']
    ticker = os.path.basename(filepath).replace('.csv', '')
    df['Ticker'] = ticker
    return df

In [4]:
test_df = load_one_file(all_files[0])
test_df.head()

,Date,Close,High,Low,Open,Volume,Ticker
0,2015-12-21,37.662884,37.930386,37.303132,37.367702,1734500,A
1,2015-12-22,38.022614,38.124082,37.561398,37.828900,1643200,A
2,2015-12-23,38.529961,38.612980,38.179436,38.391595,1510700,A
3,2015-12-24,38.871262,38.981952,38.437718,38.548412,874500,A
4,2015-12-28,38.539181,38.815913,38.299350,38.760566,1458200,A


In [5]:
all_dfs = [load_one_file(f) for f in all_files]
combined = pd.concat(all_dfs, ignore_index=True)
print(combined.shape)
combined.head()

(1220725, 7)


,Date,Close,High,Low,Open,Volume,Ticker
0,2015-12-21,37.662884,37.930386,37.303132,37.367702,1734500,A
1,2015-12-22,38.022614,38.124082,37.561398,37.828900,1643200,A
2,2015-12-23,38.529961,38.612980,38.179436,38.391595,1510700,A
3,2015-12-24,38.871262,38.981952,38.437718,38.548412,874500,A
4,2015-12-28,38.539181,38.815913,38.299350,38.760566,1458200,A


In [6]:
print(combined.isnull().sum())
print()
print("Date range:", combined['Date'].min(), "to", combined['Date'].max())
print()
print("Data type of Date column:", combined['Date'].dtype)

Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
Ticker    0
dtype: int64

Date range: 2015-12-21 to 2025-12-19

Data type of Date column: object


In [7]:
combined['Date'] = pd.to_datetime(combined['Date'])
print(combined['Date'].dtype)

datetime64[ns]


In [8]:
combined = combined.sort_values(['Ticker', 'Date']).reset_index(drop=True)
combined.head()

,Date,Close,High,Low,Open,Volume,Ticker
0,2015-12-21,37.662884,37.930386,37.303132,37.367702,1734500,A
1,2015-12-22,38.022614,38.124082,37.561398,37.828900,1643200,A
2,2015-12-23,38.529961,38.612980,38.179436,38.391595,1510700,A
3,2015-12-24,38.871262,38.981952,38.437718,38.548412,874500,A
4,2015-12-28,38.539181,38.815913,38.299350,38.760566,1458200,A


In [9]:
combined['Daily_Return'] = combined.groupby('Ticker')['Close'].pct_change()
combined.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return
0,2015-12-21,37.662884,37.930386,37.303132,37.367702,1734500,A,NaN
1,2015-12-22,38.022614,38.124082,37.561398,37.828900,1643200,A,0.009551
2,2015-12-23,38.529961,38.612980,38.179436,38.391595,1510700,A,0.013343
3,2015-12-24,38.871262,38.981952,38.437718,38.548412,874500,A,0.008858
4,2015-12-28,38.539181,38.815913,38.299350,38.760566,1458200,A,-0.008543


In [10]:
volatility = combined.groupby('Ticker')['Daily_Return'].std().reset_index()
volatility.columns = ['Ticker', 'Volatility']
volatility = volatility.sort_values('Volatility', ascending=False)
volatility.head(10)

,Ticker,Volatility
407,SNDK,0.061401
103,COIN,0.054333
39,APP,0.048508
225,HOOD,0.046749
314,MRNA,0.045336
367,PLTR,0.044708
405,SMCI,0.043344
449,TTD,0.043344
323,NCLH,0.038581
356,PCG,0.037404


In [11]:
first_last = combined.groupby('Ticker')['Close'].agg(['first', 'last']).reset_index()
first_last['Cumulative_Return_%'] = ((first_last['last'] - first_last['first']) / first_last['first']) * 100
first_last = first_last.sort_values('Cumulative_Return_%', ascending=False)
first_last.head(10)

,Ticker,first,last,Cumulative_Return_%
338,NVDA,0.802516,180.990005,22452.822523
26,AMD,2.530000,213.429993,8335.968185
48,AXON,17.930000,594.200012,3213.998896
446,TSLA,15.504000,481.200012,3003.715307
45,AVGO,11.145738,339.709991,2947.891510
32,ANET,4.472500,131.119995,2831.693674
283,LRCX,6.838094,172.270004,2419.269410
439,TPL,12.388881,299.619995,2318.458952
381,PWR,19.593523,426.660004,2077.556344
266,KLAC,57.242870,1245.670044,2076.113875


In [12]:
summary = volatility.merge(first_last[['Ticker', 'Cumulative_Return_%']], on='Ticker')
summary = summary.sort_values('Cumulative_Return_%', ascending=False).reset_index(drop=True)
summary.head(10)

,Ticker,Volatility,Cumulative_Return_%
0,NVDA,0.031422,22452.822523
1,AMD,0.037312,8335.968185
2,AXON,0.030254,3213.998896
3,TSLA,0.037356,3003.715307
4,AVGO,0.024522,2947.891510
5,ANET,0.027977,2831.693674
6,LRCX,0.027262,2419.269410
7,TPL,0.029225,2318.458952
8,PWR,0.020786,2077.556344
9,KLAC,0.025144,2076.113875


In [13]:
avg_volume = combined.groupby('Ticker')['Volume'].mean().reset_index()
avg_volume.columns = ['Ticker', 'Avg_Volume']
summary = summary.merge(avg_volume, on='Ticker')
summary.head(10)

,Ticker,Volatility,Cumulative_Return_%,Avg_Volume
0,NVDA,0.031422,22452.822523,4.588382e+08
1,AMD,0.037312,8335.968185,6.136424e+07
2,AXON,0.030254,3213.998896,7.377188e+05
3,TSLA,0.037356,3003.715307,1.155127e+08
4,AVGO,0.024522,2947.891510,2.784479e+07
5,ANET,0.027977,2831.693674,1.141773e+07
6,LRCX,0.027262,2419.269410,1.838874e+07
7,TPL,0.029225,2318.458952,2.737235e+05
8,PWR,0.020786,2077.556344,1.377794e+06
9,KLAC,0.025144,2076.113875,1.313509e+06


In [14]:
summary.to_excel("stock_summary.xlsx", sheet_name="Summary", index=False)
print("Saved.")

Saved.


In [15]:
summary.sort_values('Cumulative_Return_%').head(5)

,Ticker,Volatility,Cumulative_Return_%,Avg_Volume
500,VTRS,0.023474,-72.791175,8.080610e+06
499,PCG,0.037404,-68.159982,1.360435e+07
498,PSKY,0.030845,-64.748719,9.874746e+06
497,NCLH,0.038581,-60.975609,1.218017e+07
496,KHC,0.016912,-46.741740,7.108006e+06


In [16]:
tickers_of_interest = ['NVDA', 'AMD', 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'COIN', 'PCG', 'NCLH', 'KHC', 'META']
detail = combined[combined['Ticker'].isin(tickers_of_interest)].copy()
print(detail.shape)
detail['Ticker'].value_counts()

(28844, 8)


Ticker
AAPL     2515
AMD      2515
AMZN     2515
GOOGL    2515
KHC      2515
META     2515
MSFT     2515
NCLH     2515
NVDA     2515
PCG      2515
TSLA     2515
COIN     1179
Name: count, dtype: int64

In [17]:
detail = detail.sort_values(['Ticker', 'Date']).reset_index(drop=True)
detail['MA_50'] = detail.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=50).mean())
detail['MA_200'] = detail.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=200).mean())
detail.tail()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return,MA_50,MA_200
28839,2025-12-15,475.309998,481.769989,467.660004,469.440002,114542200,TSLA,0.035624,436.491801,346.99240
28840,2025-12-16,489.880005,491.500000,465.829987,472.209991,107608100,TSLA,0.030654,437.224401,348.01855
28841,2025-12-17,467.260010,495.279999,466.200012,488.220001,106490400,TSLA,-0.046175,437.907801,348.99465
28842,2025-12-18,483.369995,490.859985,473.119995,478.160004,95168400,TSLA,0.034478,438.801401,350.01600
28843,2025-12-19,481.200012,490.489990,474.720001,488.119995,103305400,TSLA,-0.004489,439.714601,351.10475


In [18]:
with pd.ExcelWriter("stock_summary.xlsx", engine='openpyxl', mode='a') as writer:
    detail.to_excel(writer, sheet_name="Detail_12_Tickers", index=False)
print("Saved.")

Saved.


In [19]:
detail_sorted = detail.sort_values(['Ticker', 'Date'])
detail_sorted['Indexed_100'] = detail_sorted.groupby('Ticker')['Close'].transform(lambda x: (x / x.iloc[0]) * 100)
detail_sorted.head()

,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return,MA_50,MA_200,Indexed_100
0,2015-12-21,24.199585,24.208604,23.802759,24.188311,190362400,AAPL,NaN,NaN,NaN,100.000000
1,2015-12-22,24.177036,24.287516,24.001169,24.215366,131157600,AAPL,-0.000932,NaN,NaN,99.906822
2,2015-12-23,24.488176,24.542288,24.170264,24.186047,130629600,AAPL,0.012869,NaN,NaN,101.192547
3,2015-12-24,24.357407,24.576112,24.339369,24.576112,54281600,AAPL,-0.005340,NaN,NaN,100.652167
4,2015-12-28,24.084593,24.280751,23.940293,24.258203,106816800,AAPL,-0.011200,NaN,NaN,99.524818


In [20]:
comparison_tickers = ['NVDA', 'TSLA', 'AAPL', 'KHC', 'PCG']
comparison = detail_sorted[detail_sorted['Ticker'].isin(comparison_tickers)][['Date', 'Ticker', 'Indexed_100']]
comparison_pivot = comparison.pivot(index='Date', columns='Ticker', values='Indexed_100')
comparison_pivot.head()

Ticker,AAPL,KHC,NVDA,PCG,TSLA
Date,,,,,
2015-12-21,100.000000,100.000000,100.000000,100.000000,100.000000
2015-12-22,99.906822,102.171527,100.091154,100.282678,98.877710
2015-12-23,101.192547,104.385021,100.486312,101.922327,98.770207
2015-12-24,100.652167,103.614490,100.820626,101.658466,99.144307
2015-12-28,99.524818,102.857983,100.729442,101.733844,98.447714


In [21]:
with pd.ExcelWriter("stock_summary.xlsx", engine='openpyxl', mode='a') as writer:
    comparison_pivot.to_excel(writer, sheet_name="Growth_Comparison")
print("Saved.")

Saved.


In [22]:
summary['Risk_Adjusted_Return'] = summary['Cumulative_Return_%'] / (summary['Volatility'] * 100)
summary_sorted = summary.sort_values('Risk_Adjusted_Return', ascending=False)
summary_sorted.head(10)

,Ticker,Volatility,Cumulative_Return_%,Avg_Volume,Risk_Adjusted_Return
0,NVDA,0.031422,22452.822523,4.588382e+08,7145.602884
1,AMD,0.037312,8335.968185,6.136424e+07,2234.131247
4,AVGO,0.024522,2947.891510,2.784479e+07,1202.141588
2,AXON,0.030254,3213.998896,7.377188e+05,1062.352857
5,ANET,0.027977,2831.693674,1.141773e+07,1012.163108
8,PWR,0.020786,2077.556344,1.377794e+06,999.481905
6,LRCX,0.027262,2419.269410,1.838874e+07,887.425633
9,KLAC,0.025144,2076.113875,1.313509e+06,825.699258
11,ARES,0.022523,1857.958413,8.102817e+05,824.906560
3,TSLA,0.037356,3003.715307,1.155127e+08,804.086902


In [23]:
pivot_prices = detail_sorted.pivot(index='Date', columns='Ticker', values='Close')
correlation_matrix = pivot_prices.corr()
correlation_matrix.round(2)

Ticker,AAPL,AMD,AMZN,COIN,GOOGL,KHC,META,MSFT,NCLH,NVDA,PCG,TSLA
Ticker,,,,,,,,,,,,
AAPL,1.00,0.94,0.90,0.48,0.95,-0.55,0.83,0.98,-0.77,0.83,-0.64,0.93
AMD,0.94,1.00,0.89,0.59,0.94,-0.56,0.82,0.96,-0.74,0.80,-0.64,0.87
AMZN,0.90,0.89,1.00,0.88,0.90,-0.68,0.88,0.92,-0.67,0.79,-0.73,0.86
COIN,0.48,0.59,0.88,1.00,0.69,-0.56,0.77,0.65,0.74,0.63,-0.05,0.58
GOOGL,0.95,0.94,0.90,0.69,1.00,-0.51,0.89,0.95,-0.64,0.89,-0.56,0.90
KHC,-0.55,-0.56,-0.68,-0.56,-0.51,1.00,-0.44,-0.58,0.49,-0.35,0.93,-0.49
META,0.83,0.82,0.88,0.77,0.89,-0.44,1.00,0.89,-0.46,0.95,-0.43,0.74
MSFT,0.98,0.96,0.92,0.65,0.95,-0.58,0.89,1.00,-0.74,0.87,-0.65,0.89
NCLH,-0.77,-0.74,-0.67,0.74,-0.64,0.49,-0.46,-0.74,1.00,-0.45,0.65,-0.73


In [24]:
pivot_returns = detail_sorted.pivot(index='Date', columns='Ticker', values='Daily_Return')
correlation_matrix_returns = pivot_returns.corr()
correlation_matrix_returns.round(2)

Ticker,AAPL,AMD,AMZN,COIN,GOOGL,KHC,META,MSFT,NCLH,NVDA,PCG,TSLA
Ticker,,,,,,,,,,,,
AAPL,1.00,0.44,0.57,0.38,0.61,0.29,0.52,0.68,0.34,0.54,0.15,0.44
AMD,0.44,1.00,0.44,0.43,0.42,0.12,0.38,0.45,0.27,0.60,0.09,0.36
AMZN,0.57,0.44,1.00,0.47,0.64,0.14,0.61,0.67,0.29,0.54,0.12,0.41
COIN,0.38,0.43,0.47,1.00,0.41,-0.00,0.39,0.41,0.40,0.46,0.18,0.46
GOOGL,0.61,0.42,0.64,0.41,1.00,0.21,0.61,0.71,0.34,0.54,0.15,0.39
KHC,0.29,0.12,0.14,-0.00,0.21,1.00,0.14,0.26,0.19,0.13,0.15,0.12
META,0.52,0.38,0.61,0.39,0.61,0.14,1.00,0.60,0.30,0.49,0.10,0.34
MSFT,0.68,0.45,0.67,0.41,0.71,0.26,0.60,1.00,0.32,0.62,0.17,0.42
NCLH,0.34,0.27,0.29,0.40,0.34,0.19,0.30,0.32,1.00,0.31,0.18,0.30


In [25]:
with pd.ExcelWriter("stock_summary.xlsx", engine='openpyxl', mode='a') as writer:
    summary_sorted.to_excel(writer, sheet_name="Summary_RiskAdjusted", index=False)
    correlation_matrix_returns.to_excel(writer, sheet_name="Correlation_Matrix")
print("Saved.")

Saved.
